# 基于静态Tensor的ReLU向量算子实验

ReLU是深度学习中的典型激活函数，在高性能计算教学中，适合用来观察数据切分、片上存储、数据搬运和多核并行之间的关系。本实验围绕一维向量ReLU计算展开，在Ascend C环境中使用静态Tensor编程方式实现ReLU向量算子。

本节学习大纲如下：

1. 环境准备：创建实验目录并加载CANN环境
2. 算子分析：说明输入输出、计算关系、分块策略和多核任务划分
3. 核函数开发：实现基于静态Tensor的ReLU核函数
4. 核函数运行验证：完成Host侧调用、构建运行、结果校验和性能分析
5. 实验总结：归纳静态Tensor、向量计算接口和多核分块执行之间的实现过程


---
## 1. 环境准备

首先创建实验所需目录，并尝试加载Ascend CANN环境变量。

目录划分如下：

- `Source/01.01/include`：保存Host侧公共头文件。
- `Source/01.01/src`：保存输入生成和结果校验代码。
- `Source/01.01/ascend_ops/op_kernel`：保存Device侧核函数代码。
- `Source/01.01/ascend_ops/host_launch`：保存Host侧核函数调用代码。
- `Source/01.01/scripts`：保存完整构建与运行脚本。
- `Source/01.01/results`：保存运行结果。

如果当前环境已安装CANN，下面的代码会加载环境变量。完整编译和运行需要在已配置CANN的昇腾环境中完成。

In [ ]:
!mkdir -p Source/01.01/include
!mkdir -p Source/01.01/src
!mkdir -p Source/01.01/scripts
!mkdir -p Source/01.01/results
!mkdir -p Source/01.01/ascend_ops/op_host
!mkdir -p Source/01.01/ascend_ops/host_launch
!mkdir -p Source/01.01/ascend_ops/op_kernel

import os
import subprocess
from pathlib import Path

candidate_paths = [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    os.environ.get("ASCEND_HOME_PATH"),
    os.environ.get("ASCEND_CANN_PACKAGE_PATH"),
    "/usr/local/Ascend/ascend-toolkit/latest",
    "/usr/local/Ascend/ascend-toolkit",
    "/opt/Ascend/ascend-toolkit/latest",
    "/opt/Ascend/ascend-toolkit",
    "/opt/conda/Ascend/cann-9.0.0/aarch64-linux",
    str(Path.home() / "Ascend/ascend-toolkit/latest"),
    str(Path.home() / "Ascend/ascend-toolkit"),
    "/workspace/Ascend/ascend-toolkit/latest",
    "/workspace/Ascend/ascend-toolkit",
]

set_env = None
for item in candidate_paths:
    if item and Path(item, "set_env.sh").exists():
        set_env = Path(item, "set_env.sh")
        break

if set_env is None:
    for root in [Path("/usr/local/Ascend"), Path("/opt/Ascend"), Path("/opt/conda/Ascend"), Path.home(), Path("/workspace")]:
        if root.exists():
            found = list(root.glob("**/ascend-toolkit*/set_env.sh")) + list(root.glob("**/cann-*/**/set_env.sh"))
            if found:
                set_env = found[0]
                break

if set_env is not None:
    env = subprocess.check_output(
        f"bash -l -c 'source {set_env} && env'",
        shell=True,
        text=True,
    )
    for line in env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")

print("Experiment directory:", Path("Source/01.01").resolve())

---
## 2. 算子分析

### 2.1 输入输出和计算关系

本实验实现一维向量ReLU。输入和输出是长度相同的一维向量，计算关系为：

$$y_i=\begin{cases}x_i,&x_i>0\\0,&x_i\le 0\end{cases}$$

ReLU属于逐元素计算，每个输出元素只依赖对应位置的输入元素，不依赖其他元素。因此，整体向量可以按连续区间切分给多个AI Core并行处理。


### 2.2 分块策略

完整向量通常不直接一次搬入片上存储，而是先划分为多个长度固定的数据块。每个数据块按照搬入、向量计算和写回的顺序处理。

静态Tensor方式下，片上临时空间的容量在编译期确定。运行时的数据块长度需要小于静态容量，并满足基本对齐要求。这样可以使片上空间分配、数据搬运和计算流程更加清晰。


### 2.3 多核任务划分

多核执行时，整体数据被划分为多个连续数据块，不同AI Core负责不同的数据块范围。每个核只写回自己负责的输出区间，避免多个核写同一段结果。

当输入长度不能正好整除数据块长度时，Host侧先进行尾部补齐。Device侧处理补齐后的连续数据，回到Host侧后只比较有效长度范围内的结果。

### 2.4 公共参数与校验工具

下面的公共配置用于记录实验规模、数据块长度和运行参数，并提供基础校验和统计功能。

In [ ]:
%%writefile Source/01.01/include/relu_common.h
#pragma once

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstddef>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

namespace relu {

struct Config {
    uint64_t n = 1ull << 20;
    uint32_t tile_len = 1024;
    uint32_t block_dim = 8;
    uint32_t seed = 1234;
    bool sweep = false;
    bool print_output = false;
};

struct TimingUs {
    double kernel_us = 0.0;
    double total_us = 0.0;
};

struct Metrics {
    double max_abs_error = 0.0;
    double max_rel_error = 0.0;
    uint64_t mismatch_count = 0;
};

class Timer {
public:
    Timer() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline uint64_t round_up(uint64_t n, uint64_t align) {
    if (align == 0) {
        throw std::invalid_argument("align must be > 0");
    }
    const uint64_t rem = n % align;
    if (rem == 0) {
        return n;
    }
    const uint64_t add = align - rem;
    if (n > std::numeric_limits<uint64_t>::max() - add) {
        throw std::overflow_error("round_up overflow");
    }
    return n + add;
}

inline uint32_t ceil_div_u32(uint64_t n, uint64_t d) {
    if (d == 0) {
        throw std::invalid_argument("divisor must be > 0");
    }
    const uint64_t q = (n / d) + ((n % d) != 0 ? 1ull : 0ull);
    if (q > static_cast<uint64_t>(std::numeric_limits<uint32_t>::max())) {
        throw std::overflow_error("ceil_div_u32 result exceeds uint32_t range");
    }
    return static_cast<uint32_t>(q);
}

inline size_t checked_float_bytes(uint64_t n) {
    if (n > static_cast<uint64_t>(std::numeric_limits<size_t>::max() / sizeof(float))) {
        throw std::overflow_error("float byte size overflows size_t");
    }
    return static_cast<size_t>(n) * sizeof(float);
}

inline void check_config(uint64_t n, uint32_t tile_len,
                         uint32_t dtype_size = sizeof(float)) {
    if (n == 0) {
        throw std::invalid_argument("n must be > 0");
    }
    if (tile_len == 0) {
        throw std::invalid_argument("tile_len must be > 0");
    }
    if ((static_cast<uint64_t>(tile_len) * dtype_size) % 32 != 0) {
        throw std::invalid_argument("tile_len * sizeof(T) must be a multiple of 32 bytes");
    }
}

inline double effective_gbps(uint64_t n, double total_us) {
    if (total_us <= 0.0) {
        return 0.0;
    }
    const double bytes = static_cast<double>(n) * static_cast<double>(sizeof(float)) * 2.0;
    return bytes / (total_us * 1.0e-6) / 1.0e9;
}

inline void print_header() {
    std::cout << std::setw(12) << "N"
              << std::setw(12) << "paddedN"
              << std::setw(12) << "tileLen"
              << std::setw(12) << "numTiles"
              << std::setw(10) << "cores"
              << std::setw(14) << "kernel_us"
              << std::setw(14) << "total_us"
              << std::setw(12) << "GB/s"
              << std::setw(14) << "max_abs"
              << std::setw(14) << "max_rel"
              << std::setw(10) << "errors"
              << std::setw(10) << "status"
              << "\n";
}

inline void print_result_row(uint64_t n,
                             uint64_t padded_n,
                             uint32_t tile_len,
                             uint32_t num_tiles,
                             uint32_t block_dim,
                             const TimingUs& timing,
                             const Metrics& metrics) {
    const bool pass = metrics.mismatch_count == 0;

    std::cout << std::setw(12) << n
              << std::setw(12) << padded_n
              << std::setw(12) << tile_len
              << std::setw(12) << num_tiles
              << std::setw(10) << block_dim
              << std::setw(14) << std::fixed << std::setprecision(2) << timing.kernel_us
              << std::setw(14) << std::fixed << std::setprecision(2) << timing.total_us
              << std::setw(12) << std::fixed << std::setprecision(3) << effective_gbps(padded_n, timing.total_us)
              << std::setw(14) << std::scientific << std::setprecision(3) << metrics.max_abs_error
              << std::setw(14) << std::scientific << std::setprecision(3) << metrics.max_rel_error
              << std::setw(10) << std::defaultfloat << metrics.mismatch_count
              << std::setw(10) << (pass ? "PASS" : "FAIL")
              << "\n";
}

}

---
## 3. 核函数开发

Device侧核函数运行在AI Core上，负责完成Global Memory数据访问、片上空间分配、数据搬运、ReLU计算和结果写回。

### 3.1 静态容量和任务范围

本段创建Device侧核函数文件。代码先给出静态Tensor可使用的最大容量，再根据当前AI Core编号确定该核负责的数据块范围。

In [ ]:
%%writefile Source/01.01/ascend_ops/op_kernel/relu_static_tensor.cpp
#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxTileLen = 8192;

__aicore__ inline void GetTileRange(uint32_t numTiles,
                                    uint32_t launchBlockDim,
                                    uint32_t& beginTile,
                                    uint32_t& endTile) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginTile = 0;
        endTile = 0;
        return;
    }
    const uint32_t tilesPerCore = (numTiles + coreNum - 1) / coreNum;
    beginTile = coreId * tilesPerCore;
    endTile = beginTile + tilesPerCore;
    if (endTile > numTiles) {
        endTile = numTiles;
    }
}
}

### 3.2 Tensor绑定与片上空间分配

进入核函数后，输入输出地址需要绑定为Device侧可访问的Tensor。随后在片上存储中分配临时空间，每个数据块复用这块空间完成处理。

这一部分体现片上存储复用思想：临时空间不随数据块重复申请，而是在循环中反复使用。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/op_kernel/relu_static_tensor.cpp
extern "C" __global__ __aicore__ void relu_static_tensor(GM_ADDR x,
                                                          GM_ADDR y,
                                                          uint32_t paddedLength,
                                                          uint32_t numTiles,
                                                          uint32_t tileLen,
                                                          uint32_t launchBlockDim) {
    InitSocState();

    GlobalTensor<float> xGm;
    GlobalTensor<float> yGm;
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), paddedLength);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), paddedLength);

    if (tileLen == 0 || tileLen > kStaticMaxTileLen) {
        return;
    }

    uint32_t beginTile = 0;
    uint32_t endTile = 0;
    GetTileRange(numTiles, launchBlockDim, beginTile, endTile);
    if (beginTile >= endTile) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> tileLocal = ubAllocator.Alloc<float, kStaticMaxTileLen>();
    tileLocal.SetSize(kStaticMaxTileLen);

### 3.3 搬入、计算和写回

核函数主体按数据块循环执行。每次循环只处理一段连续数据，流程可以概括为三步：先从Global Memory搬入输入数据，再在UB中调用`Maxs`完成`max(x,0)`，最后把结果写回Global Memory。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/op_kernel/relu_static_tensor.cpp
    for (uint32_t tileId = beginTile; tileId < endTile; ++tileId) {
        const uint32_t base = tileId * tileLen;

        DataCopy(tileLocal, xGm[base], tileLen);
        PipeBarrier<PIPE_ALL>();

        Maxs(tileLocal, tileLocal, 0.0f, tileLen);
        PipeBarrier<PIPE_ALL>();

        DataCopy(yGm[base], tileLocal, tileLen);
        PipeBarrier<PIPE_ALL>();
    }
}

---
## 4. 正确性验证

核函数开发完成后，需要由Host侧完成输入准备、Device内存申请、核函数启动、结果回拷、结果校验和性能统计。Device侧负责实际计算，Host侧负责组织一次完整实验。

### 4.1 Host 侧参考结果与校验函数

下面的头文件声明输入生成、参考结果和误差比较函数。用于判断Device侧输出是否符合ReLU的计算关系。


In [ ]:
%%writefile Source/01.01/include/relu_ref.h
#pragma once

#include "relu_common.h"

#include <cstdint>
#include <cstddef>
#include <vector>

namespace relu {

std::vector<float> make_input(uint64_t n, uint32_t seed);
std::vector<float> relu_reference(const std::vector<float>& x);
Metrics compare_vectors(const std::vector<float>& got,
                        const std::vector<float>& ref,
                        double atol = 0.0,
                        double rtol = 0.0);
void dump_sample(const std::vector<float>& x,
                 const std::vector<float>& y,
                 const std::vector<float>& ref,
                 size_t count = 8);

}

下面的实现生成包含正数、负数和零的输入数据，并计算标准ReLU结果。


In [ ]:
%%writefile Source/01.01/src/relu_ref.cpp
#include "relu_ref.h"

#include <algorithm>
#include <cmath>
#include <iostream>
#include <random>
#include <stdexcept>

namespace relu {

std::vector<float> make_input(uint64_t n, uint32_t seed) {
    std::mt19937 rng(seed);
    std::uniform_real_distribution<float> dist(-2.0f, 2.0f);
    std::vector<float> x(n);
    for (uint64_t i = 0; i < n; ++i) {
        x[i] = dist(rng);
    }

    if (n >= 8) {
        x[0] = -1.50f;
        x[1] = -0.25f;
        x[2] = 0.0f;
        x[3] = 0.125f;
        x[4] = 1.0f;
        x[5] = -2.0f;
        x[6] = 3.5f;
        x[7] = -0.001f;
    }
    return x;
}

std::vector<float> relu_reference(const std::vector<float>& x) {
    std::vector<float> y(x.size());
    for (size_t i = 0; i < x.size(); ++i) {
        y[i] = x[i] > 0.0f ? x[i] : 0.0f;
    }
    return y;
}

Metrics compare_vectors(const std::vector<float>& got,
                        const std::vector<float>& ref,
                        double atol,
                        double rtol) {
    if (got.size() != ref.size()) {
        throw std::invalid_argument("compare_vectors size mismatch");
    }
    Metrics metrics;
    for (size_t i = 0; i < got.size(); ++i) {
        const double g = static_cast<double>(got[i]);
        const double r = static_cast<double>(ref[i]);
        const double abs_err = std::abs(g - r);
        const double rel_err = abs_err / std::max(1.0, std::abs(r));
        metrics.max_abs_error = std::max(metrics.max_abs_error, abs_err);
        metrics.max_rel_error = std::max(metrics.max_rel_error, rel_err);
        if (abs_err > atol + rtol * std::abs(r)) {
            ++metrics.mismatch_count;
        }
    }
    return metrics;
}

void dump_sample(const std::vector<float>& x,
                 const std::vector<float>& y,
                 const std::vector<float>& ref,
                 size_t count) {
    const size_t m = std::min({count, x.size(), y.size(), ref.size()});
    std::cout << "sample(first " << m << "):\n";
    for (size_t i = 0; i < m; ++i) {
        std::cout << "  i=" << i
                  << " x=" << x[i]
                  << " y=" << y[i]
                  << " ref=" << ref[i]
                  << "\n";
    }
}

}

### 4.2 Host 侧核函数调用程序

Host侧程序负责解析运行参数、初始化运行时环境、申请Device内存、启动核函数并回收资源。下面先写入头文件、错误检查和参数解析部分。


In [ ]:
%%writefile Source/01.01/ascend_ops/host_launch/relu_npu_main.cpp
#include <acl/acl.h>
#include <aclrtlaunch_relu_static_tensor.h>

#include "relu_ref.h"

#include <algorithm>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <limits>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                                     \
    do {                                                                                    \
        aclError _ret = (expr);                                                            \
        if (_ret != ACL_SUCCESS) {                                                         \
            throw std::runtime_error(std::string("ACL error: ") + #expr +                 \
                                     ", code=" + std::to_string(static_cast<int>(_ret))); \
        }                                                                                   \
    } while (0)

struct NpuConfig : relu::Config {
    int32_t device = 0;
    uint32_t warmup = 2;
    uint32_t repeat = 10;
};

struct DeviceBuffer {
    void* ptr = nullptr;

    ~DeviceBuffer() {
        if (ptr != nullptr) {
            (void)aclrtFree(ptr);
        }
    }

    DeviceBuffer() = default;
    DeviceBuffer(const DeviceBuffer&) = delete;
    DeviceBuffer& operator=(const DeviceBuffer&) = delete;
};

struct StreamGuard {
    aclrtStream stream = nullptr;

    ~StreamGuard() {
        if (stream != nullptr) {
            (void)aclrtDestroyStream(stream);
        }
    }

    StreamGuard() = default;
    StreamGuard(const StreamGuard&) = delete;
    StreamGuard& operator=(const StreamGuard&) = delete;
};

static void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>       device id, default: 0\n"
              << "  --n <num>           total valid element count, default: 1048576\n"
              << "  --tile-len <num>    static Tensor tile length, default: 1024\n"
              << "  --block-dim <num>   AI Core launch blockDim, default: 8\n"
              << "  --warmup <num>      warmup count, default: 2\n"
              << "  --repeat <num>      repeat count, default: 10\n"
              << "  --seed <num>        random seed, default: 1234\n"
              << "  --sweep             test tileLen = 256/512/1024/2048/4096/8192\n"
              << "  --print-output      print first 8 output values\n"
              << "  -h, --help          show help\n";
}

static NpuConfig parse_args(int argc, char** argv) {
    NpuConfig cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) {
                throw std::invalid_argument("missing value after " + name);
            }
            return argv[++i];
        };
        if (arg == "--device") {
            cfg.device = std::stoi(need_value(arg));
        } else if (arg == "--n") {
            cfg.n = std::stoull(need_value(arg));
        } else if (arg == "--tile-len") {
            cfg.tile_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--block-dim") {
            cfg.block_dim = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (cfg.block_dim == 0) {
                throw std::invalid_argument("--block-dim must be positive");
            }
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
            if (cfg.repeat == 0) {
                throw std::invalid_argument("--repeat must be positive");
            }
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--sweep") {
            cfg.sweep = true;
        } else if (arg == "--print-output") {
            cfg.print_output = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

本段完成输入补齐、参考结果生成和Device内存申请。补齐后的连续数据交给Device侧处理。Device内存和stream由简单的RAII对象管理，后续步骤发生异常时也能自动释放已申请资源。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/relu_npu_main.cpp
static void run_relu_static_tensor(const NpuConfig& cfg, uint32_t tileLen) {
    relu::check_config(cfg.n, tileLen, sizeof(float));
    constexpr uint32_t kStaticMaxTileLen = 8192;
    if (tileLen > kStaticMaxTileLen) {
        throw std::invalid_argument("--tile-len exceeds static Tensor capacity 8192");
    }

    const uint64_t paddedN = relu::round_up(cfg.n, tileLen);
    const uint32_t numTiles = relu::ceil_div_u32(cfg.n, tileLen);
    if (paddedN > static_cast<uint64_t>(std::numeric_limits<uint32_t>::max())) {
        throw std::invalid_argument("padded length exceeds uint32_t kernel argument range");
    }

    const size_t paddedBytes = relu::checked_float_bytes(paddedN);

    std::vector<float> xValid = relu::make_input(cfg.n, cfg.seed);
    std::vector<float> ref = relu::relu_reference(xValid);
    std::vector<float> xPadded(paddedN, 0.0f);
    std::vector<float> yPadded(paddedN, 0.0f);
    std::copy(xValid.begin(), xValid.end(), xPadded.begin());

    DeviceBuffer xDevice;
    DeviceBuffer yDevice;
    StreamGuard stream;

    ACL_CHECK(aclrtCreateStream(&stream.stream));
    ACL_CHECK(aclrtMalloc(&xDevice.ptr, paddedBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&yDevice.ptr, paddedBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemcpy(xDevice.ptr, paddedBytes, xPadded.data(), paddedBytes, ACL_MEMCPY_HOST_TO_DEVICE));

本段启动Device侧核函数，并在同步完成后统计运行时间。随后将结果复制回Host侧，并与参考结果逐元素比较。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/relu_npu_main.cpp
    auto run_once = [&](bool timed, relu::TimingUs* timing) {
        relu::Timer totalTimer;
        ACL_CHECK(aclrtMemset(yDevice.ptr, paddedBytes, 0, paddedBytes));
        relu::Timer kernelTimer;
        ACLRT_LAUNCH_KERNEL(relu_static_tensor)(cfg.block_dim, stream.stream,
                                                xDevice.ptr, yDevice.ptr,
                                                static_cast<uint32_t>(paddedN),
                                                numTiles, tileLen, cfg.block_dim);
        ACL_CHECK(aclrtSynchronizeStream(stream.stream));
        if (timed) {
            timing->kernel_us += kernelTimer.elapsed_us();
            timing->total_us += totalTimer.elapsed_us();
        }
    };

    relu::TimingUs timing;
    for (uint32_t i = 0; i < cfg.warmup; ++i) {
        run_once(false, &timing);
    }
    for (uint32_t i = 0; i < cfg.repeat; ++i) {
        run_once(true, &timing);
    }
    timing.kernel_us /= cfg.repeat;
    timing.total_us /= cfg.repeat;

    ACL_CHECK(aclrtMemcpy(yPadded.data(), paddedBytes, yDevice.ptr, paddedBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    std::vector<float> yValid(cfg.n);
    std::copy_n(yPadded.begin(), static_cast<std::ptrdiff_t>(cfg.n), yValid.begin());
    const relu::Metrics metrics = relu::compare_vectors(yValid, ref, 0.0, 0.0);

    relu::print_result_row(cfg.n, paddedN, tileLen, numTiles, cfg.block_dim, timing, metrics);
    std::cout << "  version=relu_static_tensor"
              << ", launchBlockDim=" << cfg.block_dim
              << ", tileLen=" << tileLen
              << ", warmup=" << cfg.warmup
              << ", repeat=" << cfg.repeat
              << ", movedBytes=" << (paddedBytes * 2) << "\n";

    if (cfg.print_output) {
        relu::dump_sample(xValid, yValid, ref, 8);
    }
}

主函数组织完整运行流程：解析参数，初始化运行时环境，选择设备，执行实验并释放资源。


In [ ]:
%%writefile -a Source/01.01/ascend_ops/host_launch/relu_npu_main.cpp
int main(int argc, char** argv) {
    bool aclInitialized = false;
    bool deviceSet = false;
    int32_t device = 0;

    try {
        const NpuConfig cfg = parse_args(argc, argv);
        device = cfg.device;

        ACL_CHECK(aclInit(nullptr));
        aclInitialized = true;

        ACL_CHECK(aclrtSetDevice(cfg.device));
        deviceSet = true;

        relu::print_header();
        if (cfg.sweep) {
            for (uint32_t tileLen : {256u, 512u, 1024u, 2048u, 4096u, 8192u}) {
                run_relu_static_tensor(cfg, tileLen);
            }
        } else {
            run_relu_static_tensor(cfg, cfg.tile_len);
        }

        ACL_CHECK(aclrtResetDevice(cfg.device));
        deviceSet = false;

        ACL_CHECK(aclFinalize());
        aclInitialized = false;
        return 0;
    } catch (const std::exception& e) {
        if (deviceSet) {
            (void)aclrtResetDevice(device);
        }
        if (aclInitialized) {
            (void)aclFinalize();
        }
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}

### 4.3 CMake 构建配置

`CMakeLists.txt`用于编译Device侧核函数和Host侧调用程序。CMake会引入CANN提供的Ascend C编译配置，并链接ACL。


In [ ]:
%%writefile Source/01.01/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ascendc_static_tensor_relu LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C NPU demo" ON)

add_library(relu_ref
    src/relu_ref.cpp
)
target_include_directories(relu_ref PUBLIC include)
target_compile_options(relu_ref PRIVATE -Wall -Wextra -Wpedantic)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu")
  set(SOC_VERSION "ascend910b1" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(relu_kernels STATIC
      ascend_ops/op_kernel/relu_static_tensor.cpp
  )
  ascendc_include_directories(relu_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/ascend_ops/op_host
  )
  ascendc_compile_definitions(relu_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(relu_ascend_demo
      ascend_ops/host_launch/relu_npu_main.cpp
  )
  target_include_directories(relu_ascend_demo PRIVATE
      include
      ascend_ops/op_host
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/relu_kernels
      ${CMAKE_BINARY_DIR}/out/include/relu_kernels
  )
  target_link_directories(relu_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  )
  target_link_libraries(relu_ascend_demo PRIVATE
      relu_kernels
      relu_ref
      ascendcl
  )
  add_dependencies(relu_ascend_demo relu_kernels)
endif()

install(TARGETS relu_ascend_demo RUNTIME DESTINATION bin)
install(DIRECTORY ascend_ops DESTINATION share/ascendc_static_tensor_relu)

### 4.4 完整构建与运行脚本

运行脚本负责加载CANN环境、配置CMake、编译Ascend C核函数，并运行ReLU实验。

In [ ]:
%%writefile Source/01.01/scripts/run_ascend.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"

resolve_ascend_install_path() {
  if [[ -n "${ASCEND_INSTALL_PATH}" && -d "${ASCEND_INSTALL_PATH}" ]]; then
    return 0
  fi

  local candidates=()
  [[ -n "${ASCEND_TOOLKIT_HOME:-}" ]] && candidates+=("${ASCEND_TOOLKIT_HOME}")
  [[ -n "${ASCEND_HOME_PATH:-}" ]] && candidates+=("${ASCEND_HOME_PATH}")
  [[ -n "${ASCEND_CANN_PACKAGE_PATH:-}" ]] && candidates+=("${ASCEND_CANN_PACKAGE_PATH}")

  candidates+=(
    "${ASCEND_INSTALL_PATH_DEFAULT}"
    "/usr/local/Ascend/ascend-toolkit"
    "/opt/Ascend/ascend-toolkit/latest"
    "/opt/Ascend/ascend-toolkit"
    "${HOME}/Ascend/ascend-toolkit/latest"
    "${HOME}/Ascend/ascend-toolkit"
    "/workspace/Ascend/ascend-toolkit/latest"
    "/workspace/Ascend/ascend-toolkit"
  )

  local path
  for path in "${candidates[@]}"; do
    if [[ -n "${path}" && -d "${path}" ]]; then
      ASCEND_INSTALL_PATH="${path}"
      return 0
    fi
  done

  local found_set_env=""
  found_set_env=$(find /usr/local/Ascend /opt/Ascend "${HOME}" /workspace -path "*/ascend-toolkit*/set_env.sh" -print -quit 2>/dev/null || true)
  if [[ -n "${found_set_env}" ]]; then
    ASCEND_INSTALL_PATH="$(dirname "${found_set_env}")"
    return 0
  fi

  return 1
}
DEVICE_ID=0
RUN_MODE="npu"
BUILD_TYPE="Release"
CLEAN=0
WARMUP=2
REPEAT=10
N=1048576
TILE_LEN=1024
BLOCK_DIM=8
SEED=1234
SWEEP=0
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run_ascend.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH. If omitted, the script auto-detects ascend-toolkit.
  -v <soc>    SOC_VERSION, default: ascend910b1. Example: ascend910b2, ascend910b3, ascend310p3
  -d <id>     device id, default: 0
  -n <num>    valid element count N, default: 1048576
  -l <num>    static Tensor tile length, default: 1024
  -b <num>    AI Core launch blockDim, default: 8
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 10
  -e <num>    random seed, default: 1234
  -m <mode>   CMake run mode, default: npu. Keep npu for Ascend execution.
  -t <type>   CMake build type, default: Release
  -s          sweep tileLen = 256/512/1024/2048/4096/8192
  -c          clean build directory before building
  -h          show help

Examples:
  bash scripts/run_ascend.sh
  bash scripts/run_ascend.sh -a /usr/local/Ascend/ascend-toolkit/latest -v ascend910b1 -d 0
  bash scripts/run_ascend.sh -n 1048576 -l 1024 -b 8 -w 2 -r 10
  bash scripts/run_ascend.sh -s -n 1048576
  bash scripts/run_ascend.sh -- --print-output
USAGE
}

while getopts ":a:v:d:n:l:b:w:r:e:m:t:sch" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    n) N="${OPTARG}" ;;
    l) TILE_LEN="${OPTARG}" ;;
    b) BLOCK_DIM="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    e) SEED="${OPTARG}" ;;
    m) RUN_MODE="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    s) SWEEP=1 ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if ! resolve_ascend_install_path; then
  echo "Cannot find ascend-toolkit. Please set ASCEND_INSTALL_PATH or pass -a <path>." >&2
  echo "Tried common paths under /usr/local/Ascend, /opt/Ascend, ${HOME}, and /workspace." >&2
  exit 1
fi

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

if [[ ! -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  echo "Cannot find set_env.sh under ASCEND_INSTALL_PATH: ${ASCEND_INSTALL_PATH}" >&2
  exit 1
fi

source "${ASCEND_INSTALL_PATH}/set_env.sh"
echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}"
cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE="${RUN_MODE}"

cmake --build . -j

BIN="${BUILD_DIR}/relu_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CMD=("${BIN}" --device "${DEVICE_ID}" --n "${N}" --tile-len "${TILE_LEN}" --block-dim "${BLOCK_DIM}" --warmup "${WARMUP}" --repeat "${REPEAT}" --seed "${SEED}")
if [[ "${SWEEP}" == "1" ]]; then
  CMD+=(--sweep)
fi
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"

In [ ]:
!chmod +x Source/01.01/scripts/run_ascend.sh
!find Source/01.01 -maxdepth 3 -type f | sort

### 4.5 运行完整实验

在已安装CANN且存在NPU的环境中，执行下面命令运行完整实验。默认配置使用固定输入规模、固定数据块长度和固定AI Core数量，并包含预热和重复计时过程。


In [ ]:
!cd Source/01.01 && bash scripts/run_ascend.sh -n 1048576 -l 1024 -b 8 -w 2 -r 10 | tee results/relu_ascend_demo_result.txt

也可以执行数据块长度扫描，观察不同数据块长度对搬运粒度和运行时间的影响。

In [ ]:
!cd Source/01.01 && bash scripts/run_ascend.sh -s -n 1048576 -b 8 -w 2 -r 10 | tee results/relu_ascend_sweep_result.txt

### 4.6 结果读取与性能分析

运行脚本后，可以直接查看保存的文本结果。分析时建议关注实际输入长度、补齐后长度、数据块数量、结果校验状态、平均kernel时间和总耗时。

ReLU每个元素之间没有依赖，性能主要受数据搬运、数据块粒度、多核划分和片上向量计算接口影响。


In [ ]:
from pathlib import Path

for path in [
    Path("Source/01.01/results/relu_ascend_demo_result.txt"),
    Path("Source/01.01/results/relu_ascend_sweep_result.txt"),
]:
    print("=", path)
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:4000])
    else:
        print("not found")

## 5 实验总结

本实验实现了基于静态tensor的ReLU向量算子实验。

- 完成GM与UB之间的数据搬运和片上数据处理过程；
- 实现一维输入向量在NPU上的分块计算方式；
- 实现多个AI Core对不同tile的并行处理过程；
